In [1]:
%% 清理环境
clear; clc;

fprintf('=== 3-RRR 球面并联机构逆运动学推导（修正版，可对应论文式(9)-(13)）===\n\n');

=== 3-RRR 球面并联机构逆运动学推导（修正版，可对应论文式(9)-(13)）===

In [2]:

%% 1) 符号变量
% 机构参数
syms eta gamma alpha1 alpha2 beta real

% 主动关节角 theta_i
syms theta_i real

% 平台姿态角：theta=roll, phi=pitch, psi=yaw
syms theta phi psi real

% 用于半角代换: t = tan(theta_i/2)
syms t real

% 先把 v_i 视为已知分量（这样才能得到论文式(10)-(12) 风格）
syms vix viy viz real

% 统一化简器
simp = @(expr) simplify(expand(expr), ...
    'Steps', 100, ...
    'IgnoreAnalyticConstraints', true);

%% 2) 旋转矩阵（列向量、右手系）
Rx = @(ang) [1, 0, 0;
             0, cos(ang), -sin(ang);
             0, sin(ang),  cos(ang)];

Ry = @(ang) [ cos(ang), 0, sin(ang);
              0,        1, 0;
             -sin(ang), 0, cos(ang)];

Rz = @(ang) [cos(ang), -sin(ang), 0;
             sin(ang),  cos(ang), 0;
             0,         0,        1];

ex = sym([1;0;0]);

%% 3) u_i：对应论文式(1)(2)
R01 = simp(Rz(eta + pi/2) * Ry(pi/2 - gamma));
disp('R01 =');

R01 =

In [3]:
disp(R01);

In [4]:

u_i = simp(R01(:,1));   % x-axis
disp('u_i =');

u_i =

In [5]:
disp(u_i);

In [6]:

%% 4) w_i：按你修正后的方式
% 先绕局部 x 轴转 theta_i，再绕更新后的局部 z 轴转 alpha1
R_local = simp(Rx(theta_i) * Rz(alpha1));
disp('R_local =');

R_local =

In [7]:
disp(R_local);

In [8]:

R_w = simp(R01 * R_local);
disp('R_w =');

R_w =

In [9]:
disp(R_w);

In [10]:

w_i = simp(R_w(:,1));   % x-axis
disp('w_i =');

w_i =

In [11]:
disp(w_i);

In [12]:

%% 5) v_i 的两种写法
% 5.1 论文式(10)-(12)风格：先把 v_i 看成已知分量
v_i_sym = [vix; viy; viz];
disp('v_i_sym =');

v_i_sym =

In [13]:
disp(v_i_sym);

In [14]:

% 5.2 你的真实平台姿态写法（后面再代入）
R03 = simp(Rz(eta + 5*pi/6) * Ry(beta - pi/2));   % 150° = 5*pi/6
disp('R03 (corrected) =');

R03 (corrected) =

In [15]:
disp(R03);

In [16]:

R_rpy = simp(Rz(psi) * Ry(phi) * Rx(theta));
disp('R_rpy =');

R_rpy =

In [17]:
disp(R_rpy);

In [18]:

R_v = simp(R_rpy * R03);
disp('R_v =');

R_v =

In [19]:
disp(R_v);

In [20]:

v_i_full = simp(R_v(:,1));   % 真正展开后的 v_i
disp('v_i_full =');

v_i_full =

In [21]:
disp(v_i_full);

In [22]:

%% 6) 核心约束：w_i · v_i = cos(alpha2)
% 先用 v_i_sym 推导，得到论文风格 A,B,C
constraint_eq = simp(dot(w_i, v_i_sym) - cos(alpha2));
disp('constraint_eq =');

constraint_eq =

In [23]:
disp(constraint_eq);

In [24]:

%% 7) 先整理成 a*sin(theta_i) + b*cos(theta_i) + d = 0
% 方法：把 sin(theta_i), cos(theta_i) 视为独立基
d_term = simp(subs(constraint_eq, ...
    [sin(theta_i), cos(theta_i)], ...
    [0, 0]));

a_term = simp(subs(constraint_eq, ...
    [sin(theta_i), cos(theta_i)], ...
    [1, 0]) - d_term);

b_term = simp(subs(constraint_eq, ...
    [sin(theta_i), cos(theta_i)], ...
    [0, 1]) - d_term);

constraint_scd = simp(a_term*sin(theta_i) + b_term*cos(theta_i) + d_term);

disp('a_term ='); disp(a_term);

a_term =

In [25]:
disp('b_term ='); disp(b_term);

b_term =

In [26]:
disp('d_term ='); disp(d_term);

d_term =

In [27]:

disp('check constraint_eq - constraint_scd =');

check constraint_eq - constraint_scd =

In [28]:
disp(simp(constraint_eq - constraint_scd));

In [29]:

%% 8) 半角代换：t = tan(theta_i/2)
% sin(theta_i) = 2t/(1+t^2), cos(theta_i) = (1-t^2)/(1+t^2)
constraint_t = subs(constraint_scd, ...
    [sin(theta_i), cos(theta_i)], ...
    [2*t/(1+t^2), (1-t^2)/(1+t^2)]);
constraint_t = simp(constraint_t);

disp('constraint_t =');

constraint_t =

In [30]:
disp(constraint_t);

In [31]:

%% 9) 两边乘 (1+t^2)，整理成二次式
poly_eq = simp(expand((1 + t^2) * constraint_t));
poly_eq = collect(poly_eq, t);
poly_eq = simp(poly_eq);

disp('poly_eq =');

poly_eq =

In [32]:
disp(poly_eq);

In [33]:

%% 10) 提取 A, B, C
coef_all = coeffs(poly_eq, t, 'All');
coef_all = simp(coef_all);

% 对二次式 A*t^2 + B*t + C
if numel(coef_all) ~= 3
    error('提取到的系数个数不是 3，请检查 poly_eq 是否确实是二次多项式。');
end

A_derived = simp(coef_all(1));
B_derived = simp(coef_all(2));
C_derived = simp(coef_all(3));

disp('[A, B, C] =');

[A, B, C] =

In [34]:
disp(coef_all);

In [35]:

disp('二次项系数 A =');

二次项系数 A =

In [36]:
disp(A_derived);

In [37]:
disp('一次项系数 B =');

一次项系数 B =

In [38]:
disp(B_derived);

In [39]:
disp('常数项系数 C =');

常数项系数 C =

In [40]:
disp(C_derived);

In [41]:

%% 11) 验证 poly_eq = A*t^2 + B*t + C
check_poly = simp(poly_eq - (A_derived*t^2 + B_derived*t + C_derived));
disp('check poly_eq - (A*t^2 + B*t + C) =');

check poly_eq - (A*t^2 + B*t + C) =

In [42]:
disp(check_poly);

In [43]:

%% 12) 论文式(13)对应的求解
Delta = simp(B_derived^2 - 4*A_derived*C_derived);

t_sol_1 = simp((-B_derived + sqrt(Delta)) / (2*A_derived));
t_sol_2 = simp((-B_derived - sqrt(Delta)) / (2*A_derived));

theta_i_sol_1 = simp(2 * atan(t_sol_1));
theta_i_sol_2 = simp(2 * atan(t_sol_2));

% 更稳健的 atan2 形式
theta_i_sol_1_atan2 = simp(2 * atan2(-B_derived + sqrt(Delta), 2*A_derived));
theta_i_sol_2_atan2 = simp(2 * atan2(-B_derived - sqrt(Delta), 2*A_derived));

disp('Delta =');

Delta =

In [44]:
disp(Delta);

In [45]:

disp('t_sol_1 =');

t_sol_1 =

In [46]:
disp(t_sol_1);

In [47]:
disp('t_sol_2 =');

t_sol_2 =

In [48]:
disp(t_sol_2);

In [49]:

disp('theta_i_sol_1 =');

theta_i_sol_1 =

In [50]:
disp(theta_i_sol_1);

In [51]:
disp('theta_i_sol_2 =');

theta_i_sol_2 =

In [52]:
disp(theta_i_sol_2);

In [53]:

disp('theta_i_sol_1_atan2 =');

theta_i_sol_1_atan2 =

In [54]:
disp(theta_i_sol_1_atan2);

In [55]:
disp('theta_i_sol_2_atan2 =');

theta_i_sol_2_atan2 =

In [56]:
disp(theta_i_sol_2_atan2);

In [57]:

%% 13) 第二步：把真实的 v_i_full 代回 A,B,C
A_full = simp(subs(A_derived, [vix, viy, viz], transpose(v_i_full)));
B_full = simp(subs(B_derived, [vix, viy, viz], transpose(v_i_full)));
C_full = simp(subs(C_derived, [vix, viy, viz], transpose(v_i_full)));

disp('A_full (substitute v_i_full) =');

A_full (substitute v_i_full) =

In [58]:
disp(A_full);

In [59]:
disp('B_full (substitute v_i_full) =');

B_full (substitute v_i_full) =

In [60]:
disp(B_full);

In [61]:
disp('C_full (substitute v_i_full) =');

C_full (substitute v_i_full) =

In [62]:
disp(C_full);

In [63]:

%% 14) 再用 full 形式验证一次
constraint_eq_full = simp(dot(w_i, v_i_full) - cos(alpha2));
poly_eq_full = simp(subs(A_full*t^2 + B_full*t + C_full, t, t));

constraint_t_full = subs(constraint_eq_full, ...
    [sin(theta_i), cos(theta_i)], ...
    [2*t/(1+t^2), (1-t^2)/(1+t^2)]);
constraint_t_full = simp(constraint_t_full);

poly_eq_full_from_constraint = simp(expand((1+t^2) * constraint_t_full));
poly_eq_full_from_constraint = collect(poly_eq_full_from_constraint, t);
poly_eq_full_from_constraint = simp(poly_eq_full_from_constraint);

disp('poly_eq_full_from_constraint =');

poly_eq_full_from_constraint =

In [64]:
disp(poly_eq_full_from_constraint);

In [65]:

disp('check full polynomial consistency =');

check full polynomial consistency =

In [66]:
disp(simp(poly_eq_full_from_constraint - poly_eq_full));

In [67]:

fprintf('\n=== 推导完成 ===\n');

=== 推导完成 ===

In [68]:
fprintf('说明：\n');

说明：

In [69]:
fprintf('1) A_derived, B_derived, C_derived 是论文式(10)-(12)风格；\n');

1) A_derived, B_derived, C_derived 是论文式(10)-(12)风格；

In [70]:
fprintf('2) A_full, B_full, C_full 是把真实 v_i_full 代回后的完整表达；\n');

2) A_full, B_full, C_full 是把真实 v_i_full 代回后的完整表达；

In [71]:
fprintf('3) theta_i_sol_1/2 对应论文式(13)的两组解。\n');

3) theta_i_sol_1/2 对应论文式(13)的两组解。